<a href="https://colab.research.google.com/github/LailaBulh/Ingenieria_de_Datos_Avanzada/blob/main/Titanic_PySpark_LB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Operaciones básicas con DataFrames en PySpark (Titanic)**



**Nombre:** Laila Montserrat Bulhosen Ramos **Matrícula:** 263166

**Docente:** Dr. Vicente García Jiménez

**Materia:** Ingeniería de Datos Avanzada

**Link Github:** [df_Titanic_PySpark](https://github.com/LailaBulh/Ingenieria_de_Datos_Avanzada/blob/main/Titanic_PySpark_LB.ipynb)

**Fecha:** Mayo 2026

## **Instalación y configuración de ambiente**

### **Carga de librerías**

In [8]:
### Clase para crear sesión en Spark
from pyspark.sql import SparkSession

### Módulo de funciones SQL de PySpark
### Se importa como F para facilitar su uso
from pyspark.sql import functions as F


### **Instalación de PySpark**


In [11]:
### Verificar si PySpark está instalado

try:
  import pyspark

  print('PySpark ya esta instalado')
  print(f'Version de PySpark instalada: {pyspark.__version__}')


except ModuleNotFoundError:
    print("PySpark no está instalado. Instalando...")
    !pip install pyspark -q

    import pyspark
    print("Instalación completada")
    print("Versión:", pyspark.__version__)

PySpark ya esta instalado
Version de PySpark instalada: 4.0.2


In [13]:
### MOSTRAR VERSIÓN DESDE TERMINAL

print("\nInformación desde terminal:")
!pyspark --version



Información desde terminal:
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 4.0.2
      /_/
                        
Using Scala version 2.13.16, OpenJDK 64-Bit Server VM, 17.0.18
Branch HEAD
Compiled by user runner on 2026-02-02T08:08:13Z
Revision 7cc3b9bcdaab8c923f23cdbc9ce922530e1becf1
Url https://github.com/apache/spark
Type --help for more information.


In [14]:
spark = (
    SparkSession.builder

    ### Nombre visible en la Spark UI y en los logs del clúster
    .appName('Titanic_PySpark')

    ### Número de particiones en operaciones de shuffle (default: 200)
    ### 200 particiones para datasets pequeños es excesivo y lento
    ### Regla general: 2-4 particiones por core en el clúster
    .config('spark.sql.shuffle.partitions', '8')

    ### Si ya existe una sesión activa, la reutiliza (no crea una nueva)
    .getOrCreate()
)

print(f'   SparkSession lista')
print(f'   Nombre app       : {spark.sparkContext.appName}')
print(f'   Master           : {spark.sparkContext.master}')
print(f'   Cores disponibles: {spark.sparkContext.defaultParallelism}')

   SparkSession lista
   Nombre app       : Titanic_PySpark
   Master           : local[*]
   Cores disponibles: 2


## **1. Carga de datos usando '*spark.read.csv* '**

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
titanic_data = '/content/drive/MyDrive/Titanic-Dataset.csv'


In [15]:
### Dirección donde se ubica el archivo en drive

titanic_csv = '/content/drive/MyDrive/Titanic-Dataset.csv'


### Cargar el archivo CSV en un DataFrame de PySpark
titanic_df = (
    spark.read

    ### Indica que el archivo contiene encabezados
    .option('header', 'true')

    ### Aplicar esquema automáticamente
    .option('inferSchema', 'true')

    ### Cargar archivo CSV
    .csv(titanic_csv)
)

## **2. Exploración inicial**

In [18]:
### Muestra las primeras filas con show()

print('Primeras 10 filas del dataset Titanic')
titanic_df.show(10)

Primeras 10 filas del dataset Titanic
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|  22|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|  38|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|  26|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|  35|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|  35|    0|    0|          373450|   8.05| NULL|       S|
|          6|       0|     3|    Moran, Mr

In [19]:
### Visualiza el esquema con printSchema()

print('Esquema del dataframe Titanic:')
titanic_df.printSchema()

Esquema del dataframe Titanic:
root
 |-- PassengerId: string (nullable = true)
 |-- Survived: string (nullable = true)
 |-- Pclass: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- SibSp: string (nullable = true)
 |-- Parch: string (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: string (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



In [21]:
### Como información adicional se muestran el total de filas y columnas dentro del dataset

print('*** Tamaño del dataset Titanic ***\n')

print(f'Número total de filas: {titanic_df.count():,} ')

print(f'Número total de columnas: {len(titanic_df.columns)}')

*** Tamaño del dataset Titanic ***

Número total de filas: 891 
Número total de columnas: 12


## **3. Selección de columnas**

Selección de las columnas *'Name', 'Age', 'Sex'*

In [30]:
df_select = titanic_df.select('Name', 'Age', 'Sex')

print('*** Resultados de seleccionar las columnas "Name", "Age", "Sex" ***\n')
df_select.show(10)

*** Resultados de seleccionar las columnas "Name", "Age", "Sex" ***

+--------------------+----+------+
|                Name| Age|   Sex|
+--------------------+----+------+
|Braund, Mr. Owen ...|  22|  male|
|Cumings, Mrs. Joh...|  38|female|
|Heikkinen, Miss. ...|  26|female|
|Futrelle, Mrs. Ja...|  35|female|
|Allen, Mr. Willia...|  35|  male|
|    Moran, Mr. James|NULL|  male|
|McCarthy, Mr. Tim...|  54|  male|
|Palsson, Master. ...|   2|  male|
|Johnson, Mrs. Osc...|  27|female|
|Nasser, Mrs. Nich...|  14|female|
+--------------------+----+------+
only showing top 10 rows


## **4. Filtrado de datos**

Filtra los pasajeros:
* Mayores de 18 años
* De sexo femenino


In [36]:
### Se crea un dataframe para guardar los resultados del filtro

df_filter = df_select.filter(
                              (df_select.Age > 18)
                              & (df_select.Sex == 'female')
)

print('*** Resultados de pasajeros femeninos mayores de 18 años ***\n')
df_filter.show(10)


*** Resultados de pasajeros femeninos mayores de 18 años ***

+--------------------+---+------+
|                Name|Age|   Sex|
+--------------------+---+------+
|Cumings, Mrs. Joh...| 38|female|
|Heikkinen, Miss. ...| 26|female|
|Futrelle, Mrs. Ja...| 35|female|
|Johnson, Mrs. Osc...| 27|female|
|Bonnell, Miss. El...| 58|female|
|Hewlett, Mrs. (Ma...| 55|female|
|Vander Planke, Mr...| 31|female|
|Asplund, Mrs. Car...| 38|female|
|Ahlin, Mrs. Johan...| 40|female|
|Turpin, Mrs. Will...| 27|female|
+--------------------+---+------+
only showing top 10 rows


## **5. Valores únicos**

Se obtienen los valores unicos de la columnas *'Pclass'*

In [39]:
### Se crea un dataframe para guardar los resultados de los valores únicos

pclass_distinct = titanic_df.select('Pclass').distinct()

print('*** Valores únicos de la columna "Pclass" ***\n')
pclass_distinct.show()

*** Valores únicos de la columna "Pclass" ***

+------+
|Pclass|
+------+
|     1|
|     3|
|     2|
+------+



## **6. Agrupación**

Total de pasajeros por *'Pclass'* y *'Sex'*

In [50]:
### Se crea un dataframe para guardar los resultados de la agrupación

### Pasajeros por clase y género
df_group = titanic_df.groupBy('Pclass', 'Sex').count()

print('*** Total de pasajeros por "Pclass" y "Sex" ***\n')
df_group.show()

### Pasajeros por clase
df_group_class = titanic_df.groupBy('Pclass').count()

print('*** Total de pasajeros por "Pclass" ***\n')
df_group_class.show()

### Pasajeros por género
df_group_sex = titanic_df.groupBy('Sex').count()

print('*** Total de pasajeros por "Sex" ***\n')
df_group_sex.show()

*** Total de pasajeros por "Pclass" y "Sex" ***

+------+------+-----+
|Pclass|   Sex|count|
+------+------+-----+
|     3|  male|  347|
|     1|female|   94|
|     3|female|  144|
|     2|female|   76|
|     2|  male|  108|
|     1|  male|  122|
+------+------+-----+

*** Total de pasajeros por "Pclass" ***

+------+-----+
|Pclass|count|
+------+-----+
|     1|  216|
|     3|  491|
|     2|  184|
+------+-----+

*** Total de pasajeros por "Sex" ***

+------+-----+
|   Sex|count|
+------+-----+
|female|  314|
|  male|  577|
+------+-----+



## **7. Estadística básica**

Aplica describe() sobre las variables numéricas


In [53]:
### Para poder aplicar describe() se identifican las columnas numéricas relevantes

cols = ['Survived','Age', 'SibSp', 'Parch', 'Fare']

print('*** Estadísticas básicas descriptivas de columnas numéricas ***\n')
titanic_df.select(cols).describe().show()

*** Estadísticas básicas descriptivas de columnas numéricas ***

+-------+-------------------+------------------+------------------+-------------------+-----------------+
|summary|           Survived|               Age|             SibSp|              Parch|             Fare|
+-------+-------------------+------------------+------------------+-------------------+-----------------+
|  count|                891|               714|               891|                891|              891|
|   mean| 0.3838383838383838| 29.69911764705882|0.5230078563411896|0.38159371492704824| 32.2042079685746|
| stddev|0.48659245426485753|14.526497332334035|1.1027434322934315| 0.8060572211299488|49.69342859718089|
|    min|                  0|              0.42|                 0|                  0|                0|
|    max|                  1|                 9|                 8|                  6|             93.5|
+-------+-------------------+------------------+------------------+-------------------+

## **8. Valores nulos**

Identificar en qué columnas hay valores nulos (usa isNull() o funciones vistas)


In [64]:
### Con un ciclo for se itera cada columna y se cuentan los valores nulos
### guardando esta información en una lista llamda null_values

null_values = [(c, titanic_df.filter(f"{c} IS NULL").count()) for c in titanic_df.columns]

### Se convierte en dataframe para una visualización final clara
nulos_df = spark.createDataFrame(null_values, ["Columna", "Nulos"])

### Se imprimen resultados
print('*** Valores nulos por columna ***\n')
nulos_df.show(truncate=False)

*** Valores nulos por columna ***

+-----------+-----+
|Columna    |Nulos|
+-----------+-----+
|PassengerId|0    |
|Survived   |0    |
|Pclass     |0    |
|Name       |0    |
|Sex        |0    |
|Age        |177  |
|SibSp      |0    |
|Parch      |0    |
|Ticket     |0    |
|Fare       |0    |
|Cabin      |687  |
|Embarked   |2    |
+-----------+-----+

